Proof of <a class="ProveItLink" href="../../../../../../_theory_nbs_/theory.ipynb">proveit</a>.<a class="ProveItLink" href="../../../../../_theory_nbs_/theory.ipynb">physics</a>.<a class="ProveItLink" href="../../../../_theory_nbs_/theory.ipynb">quantum</a>.<a class="ProveItLink" href="../../theory.ipynb">QEC2</a>.<a class="ProveItLink" href="../../theorems.ipynb#buf_gen_graph_completeness">buf_gen_graph_completeness</a> theorem
========

In [ ]:
import proveit
theory = proveit.Theory() # the theorem's theory
from proveit import defaults, display_provers # useful imports
from proveit                 import b, e, f, g, k, p, s, X, Y, ExprTuple, Function, Lambda, Variable
from proveit.core_expr_types import f_1_to_k
from proveit.logic           import And, Equals, Exists, Forall, InSet
from proveit.logic.sets      import Card, EmptySet, Set, SubsetEq
from proveit.graphs          import EdgeSequenceOf, PathsOf
from proveit.numbers         import zero, one, LessEq

from proveit.physics.quantum.QEC2 import (
        _nu, _start, AllStatesGraph, BufiloGeneratingGraph, Errors,
        Faults, IrreducibleBufiloSets, RealizationSets, s_prime, State)

In [ ]:
%proving buf_gen_graph_completeness

##### We are considering all “irreducible BUFILOS” $b$, so we add that assumption to the default assumptions:

In [ ]:
defaults.assumptions = buf_gen_graph_completeness.all_conditions()

In [ ]:
b_in_ibufs = buf_gen_graph_completeness.all_conditions()[0]

We proceed using a proof by induction on the number of faults $k$ used to generate a path from the START state $\{\ell\}$ (on the way to the END state $\emptyset$), with the goal of proving the following sub-theorem:

Given the “RealizationSets” approach, the inductive statement would then become something like this:

$\forall_{k \le |b|}\big[
\exists_{p \in \text{PathsOf}(G_{\ell \nu}^{BUF}, \text{START}\rightarrow)}\big[
\exists_{e \subseteq b | |e|=k}\big[e \in \text{RealizationSets}(\text{EdgeSeq}(p), G_{\ell \nu}^{BUF})
\big]\big]\big]$


In [ ]:
# sub_thm = (
#     Forall(k,
       
#        Exists(p,

#        Exists(e, 
#               InSet(e, RealizationSets(EdgeSequenceOf(p, BufiloGeneratingGraph), BufiloGeneratingGraph)), # end exists f1..fk
#        conditions=[SubsetEq(e, b), Equals(Card(e), k)]).with_wrapping(),
#        domain=PathsOf(BufiloGeneratingGraph, begin=_start)).with_wrapping(), # end exists p
       
#     conditions=[LessEq(k, Card(b))]).with_wrapping() # end Forall k
# )

In [ ]:
sub_thm = (
    Forall(k,
       
       Exists((p, e),

       InSet(e, RealizationSets(EdgeSequenceOf(p, BufiloGeneratingGraph), BufiloGeneratingGraph)), # end exists f1..fk
              
       conditions=[InSet(p, PathsOf(BufiloGeneratingGraph, begin=_start)), SubsetEq(e, b), Equals(Card(e), k)]).with_wrapping(), # end exists p
       
    conditions=[LessEq(k, Card(b))]).with_wrapping() # end Forall k
)

##### As a convenience in proving that sub-theorem (and eventually applying a finite induction theorem), we define the following (Lambda-defined) propositional function:

In [ ]:
sub_thm.instance_expr

In [ ]:
P = Lambda(k, sub_thm.instance_expr)

##### We want to show that $P(0)$ is TRUE, and that for $k < |b|$ we have $P(k) \Rightarrow P(k+1)$.

#### Base Case: $P(0)$

In [ ]:
P0 = P.apply(zero)

##### The proposed path is the trivial (_i.e._, single-vertex) path $p = (\text{START})$, with corresponding empty set of faults $\emptyset \subseteq b$ forming the empty sequence of faults $()$ such that $() \in \text{Realizations}(\text{EdgeSeq}(p), G_{\ell \nu}^{BUF}) = \text{Realizations}((), G_{\ell \nu}^{BUF}) \equiv \{()\}$.

In [ ]:
trivial_path = ExprTuple(_start)

We need to establish that the trivial path $(\text{START})$ is indeed a path in our BUFILO-generating graph $G_{\ell, \nu}^{BUF}$.

In [ ]:
from proveit.physics.quantum.QEC2 import buf_gen_graph_def
buf_gen_graph_def

In [ ]:
from proveit.physics.quantum.QEC2 import buf_gen_graph_is_graph
buf_gen_graph_is_graph

The vertex set for the BUFILO-generating graph $G_{\ell v}^{BUF}$ is the set of all observable sets $\text{ObsSets}_{\ell}$, but we need to make this explicit, and the vertices of an explicitly defined graph of the form $\text{Graph}(V, E)$ are obtained as follows (some opportunities for automation here):

In [ ]:
from proveit.graphs import vertices_def
vertices_def

In [ ]:
from proveit import E, V
from proveit.physics.quantum.QEC2 import ObservableSets
vertices_are_obs_sets = vertices_def.instantiate({V:ObservableSets, E:buf_gen_graph_def.rhs.operands[1]})

In [ ]:
vertices_are_obs_sets.inner_expr().lhs.operand.substitute(buf_gen_graph_def.derive_reversed())

In [ ]:
from proveit.physics.quantum.QEC2 import start_is_obs_set
start_is_obs_set

In [ ]:
from proveit.graphs import Vertices
InSet(_start, Vertices(BufiloGeneratingGraph)).prove()

In [ ]:
from proveit.graphs import WalksOf
InSet(trivial_path, WalksOf(BufiloGeneratingGraph)).prove()

In [ ]:
EdgeSequenceOf(trivial_path, BufiloGeneratingGraph).simplification()

In [ ]:
from proveit.physics.quantum.QEC2 import RealizationSets
RealizationSets(ExprTuple(), BufiloGeneratingGraph).simplification()

In [ ]:
from proveit.physics.quantum.QEC2 import realization_sets_of_empty_edge_seq
realization_sets_of_empty_edge_seq

In [ ]:
from proveit import G
realization_sets_of_empty_edge_seq.instantiate({G:BufiloGeneratingGraph})

In [ ]:
InSet(EmptySet, Set(EmptySet)).prove()

In [ ]:
InSet(trivial_path, PathsOf(BufiloGeneratingGraph, begin=_start)).prove()

In [ ]:
P0.conclude_via_example((ExprTuple(_start), EmptySet))

#### Inductive Step: $P(k) \Rightarrow P(k+1)$

$P(k)$ is given by:

In [ ]:
Pk = P.apply(k)

We assume $P(k)$ by adding $P(k)$ to the default assumptions:

In [ ]:
defaults.assumptions = defaults.assumptions + [Pk]

We Skolemize the path $p_k$ and error $e_k$ associated with our inductive hypothesis $P(k)$:

In [ ]:
from proveit.physics.quantum.QEC2 import ek, pk
pk_in_paths, ek_subseteq_b, card_ek_eq_k, ek_in_realizations = Pk.choose(pk, ek)

We then “unfold” what it means for $p_k$ to be such a path, so that we can eventually Skolemize the nodes of the path.

In [ ]:
pk_in_walks = pk_in_paths.definition().derive_right_via_equality()

In [ ]:
pk_in_walks_of = InSet(pk, WalksOf(BufiloGeneratingGraph, begin=_start)).prove()

In [ ]:
pk_in_walks_of_unfolded = pk_in_walks_of.existential_unfold(0)

In [ ]:
from proveit.numbers import one, Add
k_in_natural_pos, exist_verts_for_pk = pk_in_walks_of_unfolded.choose(k)

##### We then Skolemize those vertices $v_0, v_1, \ldots, v_k$ to $s_0, s_1, \ldots, s_k$, giving us a $k$-edge path $p = (s_0, s_1, \ldots, s_k)$, where each node $s_{i}$ is a set of observables:

In [ ]:
claims_for_s = exist_verts_for_pk.choose(s)

In [ ]:
pk_tuple = claims_for_s

In [ ]:
for item in defaults.assumptions:
    display(item)

##### We want to access or otherwise make use of the Vertices(G) membership claims produced above.

In [ ]:
s0_to_sk_in_verts = claims_for_s[0]

In [ ]:
# the proof process produces an explicit conjunction
conj = s0_to_sk_in_verts.prove()

In [ ]:
from proveit.physics.quantum.QEC2 import s_0, s_1, s_k
# InSet(s_0, Vertices(BufiloGeneratingGraph)).prove()

In [ ]:
# s0_to_sk_in_verts.partition()

In [ ]:
conj

In [ ]:
# would be nice to add a demo for this approach/trick
from proveit.numbers import one, num, subtract
conj.inner_expr().operands[0].split(subtract(k, one)).derive_any(1)

In [ ]:
pk_eq_tuple = claims_for_s[1].derive_any(0)

In [ ]:
s0_eq_start = claims_for_s[1].derive_any(2)

#### 1st Lemma: `start_path_end_points` and its Instantiation

The `start_path_end_points` theorem/lemma expresses each START-beginning path as the symmetric difference of the observable sets of a sequence of faults.

In [ ]:
from proveit.physics.quantum.QEC2 import start_path_end_points
start_path_end_points

We need to establish that:

$p_k \in \text{PathsOf}(G, \text{START}\to s_k)$,

whereas right now we know that:

$p_k \in \text{PathsOf}(G, \text{START}\to)$

In [ ]:
from proveit.graphs.walks import explicit_path_with_begin_is_path_with_ends
explicit_path_with_begin_is_path_with_ends

In [ ]:
pk_in_paths

In [ ]:
s0_eq_start.sub_left_side_into(pk_in_paths)

Now we can establish that $p_k$ is a path from $s_0$ to $s_k$, and then conclude $p_k$ is a path from $\text{START}$ to $s_{k}$.

In [ ]:
from proveit import n, s, v, G
pk_tuple_in_paths_so_to_sk = explicit_path_with_begin_is_path_with_ends.instantiate(
        {G:BufiloGeneratingGraph, n:k, v:s}
)

In [ ]:
s0_eq_start.sub_right_side_into(pk_tuple_in_paths_so_to_sk)

And finally we get to use our `start_path_end_points` theorem/lemma to give a symmetric difference formula for node $s_{k}$:

In [ ]:
from proveit import t
from proveit.physics.quantum.QEC2 import s_k
start_path_end_points_inst = start_path_end_points.instantiate({t:s_k, p:pk}, num_forall_eliminations=2)

In [ ]:
# here we don't want to assume that n = k, so we pick m
# But we are going to need to constrain m eventually to be less than the "n"
# that appears in Lemma 2 below. Perhaps there we can instantiate with n = |b|?
# and then somehow make clear that m ≤ |b|?
from proveit import m
m_in_nat, exists_f1_to_fm = start_path_end_points_inst.choose(m)

In [ ]:
f_1_to_m_in_faults, f_1_to_m_in_realizations, sk_eq_sym_diff = exists_f1_to_fm.choose()

#### 2nd Lemma: `buf_path_end_point` and its Instantiation

In [ ]:
from proveit.physics.quantum.QEC2 import buf_path_end_point
buf_path_end_point

To instantiate that for our irreducible BUFILO $b$ we need to express $b$ as a set of faults. By definition, we know $b \in \text{ERRS}$:

In [ ]:
b_in_ibufs_unfolded = b_in_ibufs.unfold()

In [ ]:
b_in_bufs = b_in_ibufs_unfolded.expr.operands[0]

In [ ]:
b_in_bufs_unfolded = b_in_bufs.unfold()

In [ ]:
b_in_errs = b_in_bufs_unfolded.operands[0]

In [ ]:
b_in_errs_unfolded = b_in_errs.unfold()

In [ ]:
from proveit import n
n_in_nat, exist_faults = b_in_errs_unfolded.choose(n)

In [ ]:
fs_in_faults, b_is_seq_faults = exist_faults.choose(f)

In [ ]:
buf_obs_set_is_empty = buf_path_end_point.instantiate({n:n, f:f, b:b})

In [ ]:
obs_fs_partitioned = buf_obs_set_is_empty.lhs.operands[1].partition(m, assumptions=defaults.assumptions + [LessEq(m, n)])

In [ ]:
buf_obs_set_is_empty_partitioned = buf_obs_set_is_empty.inner_expr().lhs.operands[1].substitute(obs_fs_partitioned, assumptions=defaults.assumptions + [LessEq(m, n)])

In [ ]:
# buf_obs_set_is_empty_partitioned.lhs.association(0, 2, assumptions=defaults.assumptions + [LessEq(m, n)])

In [ ]:
buf_obs_set_is_empty_assoc_01 = (
    buf_obs_set_is_empty_partitioned.inner_expr().lhs.associate(
        0, 2, assumptions=defaults.assumptions + [LessEq(m, n)])
)

In [ ]:
buf_obs_set_is_empty_assoc_02 = (
    buf_obs_set_is_empty_assoc_01.inner_expr().lhs.associate(
        1, 1, assumptions=defaults.assumptions + [LessEq(m, n)]))

In [ ]:
from proveit.logic.sets.symmetric_difference import empty_sym_diff_imp_set_equality
empty_sym_diff_imp_set_equality

In [ ]:
from proveit import A, B
sym_diffs_eq = empty_sym_diff_imp_set_equality.instantiate(
    {A: buf_obs_set_is_empty_assoc_02.lhs.operands[0],
     B: buf_obs_set_is_empty_assoc_02.lhs.operands[1]},
    assumptions=defaults.assumptions + [LessEq(m, n)])

In [ ]:
sk_eq_sym_diff

In [ ]:
sk_eq_other_sym_diff = sk_eq_sym_diff.sub_left_side_into(sym_diffs_eq,
            assumptions=defaults.assumptions + [LessEq(m, n)])

In [ ]:
from proveit import x
membership_eq = sk_eq_other_sym_diff.substitution(Lambda(X, InSet(x, X)))

In [ ]:
from proveit.logic.sets.symmetric_difference import sym_diff_all_as_sym_diff
sym_diff_all_as_sym_diff

In [ ]:
from proveit import f, i, j, k, x, A, S, IndexedVar, Lambda
from proveit.numbers import Interval
from proveit.physics.quantum.QEC2 import _ell, ObservableSet
m_plus_1 = Add(m, one)
_A_sub = Lambda(i, ObservableSet(Set(IndexedVar(f, i)), _ell))
_j_sub = m_plus_1
_k_sub = n
sym_diff_all_eq_sym_diff = sym_diff_all_as_sym_diff.instantiate({A:_A_sub, j:_j_sub, k:_k_sub})

In [ ]:
sym_diff_all_eq_sym_diff.substitution(Lambda(s, InSet(x, s)))

In [ ]:
# from proveit.numbers import NaturalPos
# InSet(subtract(n, m), NaturalPos).prove(assumptions=defaults.assumptions + [LessEq(m, n), InSet(x, s_k)])

In [ ]:
# ------------------------------------------ #
# Likely don't need this cell/step anymore.  #
# See below for new theorem/instantiation    #
# approach.                                  #
# ------------------------------------------ #
# elem_in_sk_impl_elem_in_other_sym_diff.consequent.unfold(
#     assumptions=defaults.assumptions + [LessEq(m, n), InSet(x, s_k)]).as_implication(InSet(x, s_k))

In [ ]:
from proveit.logic.sets.symmetric_difference import sym_diff_all_membership_existential_unfolding
sym_diff_all_membership_existential_unfolding

In [ ]:
from proveit import f, i, x, A, S, IndexedVar, Lambda
from proveit.numbers import Interval
from proveit.physics.quantum.QEC2 import _ell, ObservableSet
m_plus_1 = Add(m, one)
_A_sub = Lambda(i, ObservableSet(Set(IndexedVar(f, i)), _ell))
_S_sub = Interval(m_plus_1, n)
x_in_sk_impl_x_in_remaining_obs = sym_diff_all_membership_existential_unfolding.instantiate(
        {A:_A_sub, S:_S_sub, x:x},
        assumptions = defaults.assumptions + [LessEq(m, n), InSet(x, s_k)]
)

As a likely better alternative to the instantiation above, we consider replacing $x$ with $\nu(s_k)$ …

In [ ]:
from proveit.physics.quantum.QEC2 import nu_is_choice_fxn
nu_is_choice_fxn

In [ ]:
nu_s_k_in_s_k = nu_is_choice_fxn.instantiate({s:s_k})

In [ ]:
_A_sub = Lambda(i, ObservableSet(Set(IndexedVar(f, i)), _ell))
_S_sub = Interval(m_plus_1, n)
nu_s_k_in_remaining_obs = sym_diff_all_membership_existential_unfolding.instantiate(
        {A:_A_sub, S:_S_sub, x:Function(_nu, s_k)},
        # assumptions = defaults.assumptions + [LessEq(m, n), InSet(Function(_nu, s_k), s_k)]
        assumptions = defaults.assumptions + [LessEq(m, n)]
)

In [ ]:
j_in_m_plus_1_to_n, nu_s_k_in_obs_f_j = nu_s_k_in_remaining_obs.choose(j)

<div style="background-color: #f5f5f6; padding: 10px; border-left: 5px solid #0076a3;">
    <h4>A quick review of the status of the proof thus far</h3>
    


1. We have assumed that $P(k)$ is True — i.e., we have assumed that:$$\exists_{p \in \text{PathsOf}(G, \text{START}\to), e\subseteq b \,|\, |e|=k}\big(e \in \text{RealizationSets}_{G}(\text{EdgeSeqOf}(p, G))\big)$$
2. We eventually Skolemized the $k$-edge path $p$ to obtain an explicit path $p = (s_0, s_1, \ldots, s_k)$, where the $s_0, s_1, \ldots, s_k$ are nodes in our graph $G$ with $s_0 = \text{START}$, and we Skolemized the associated error $e$ (_i.e._, a set of faults) to $e_k \subseteq b$ with $|e_k| = k$ and $e_k \in \text{RealizationSets}_{G}(\text{EdgeSeqOf}(p, G))$.</br></br>
3. We took a side-step to show that not only is $p_k \in \text{PathsOf}(G, \text{START}\to)$, but that in fact $p_k \in \text{PathsOf}(G, \text{START}\to s_{k})$.</br></br>
4. That then allowed us to invoke a useful lemma to conclude that: $$\exists_{n \in \mathbb{N}}\Big[\exists_{f_1,f_2,\ldots,f_n \in \text{FAULTS}}\,|\,(f_1,f_2,\ldots,f_n)\in\text{RealizationSets}_{G}(\text{EdgeSeqOf}(p_k, G))$$
$$\Big(s_k = \text{START}\triangle\text{Obs}_{\ell}(\{f_1\})\triangle\text{Obs}_{\ell}(\{f_2\})\triangle\ldots\triangle\text{Obs}_{\ell}(\{f_n\})\Big)\Big]$$
5. We then Skolemized $n$ and $f$ in the existential claim , taking $\{n:m, f:f\}$ to derive:<br/>
<ul><li>$f_{1}, f_{2}, \ldots, f_{m} \in \text{FAULTS}$</li>
    <li>$(f_{1}, f_{2}, \ldots, f_{m}) \in \text{RealizationSets}_{G}(\text{EdgeSeqOf}(p_k, G))$</li>
    <li>$s_k = \text{START}\triangle\text{Obs}_{\ell}(\{f_1\})\triangle\text{Obs}_{\ell}(\{f_1\})\triangle\cdots\triangle\text{Obs}_{\ell}(\{f_m\})$</li></ul>

6. From $b$ being a BUFILO, we know that $b \in \text{ERRS}$ and thus $b$ is a set of faults. Since $e \subseteq b$, we know the set of faults $b$ includes the set of faults $e$. We skolemize an existence claim to obtain $b = \{f_1, f_2, \ldots, f_n\}$, where $m \le n$.<br/><br/>
7. We then utilize a second lemma characterizing the logical effect of a BUFILO, to conclude that: $$\text{START}\triangle\text{Obs}_{\ell}(\{f_1\})\triangle\text{Obs}_{\ell}(\{f_2\})\triangle\cdots\triangle\text{Obs}_{\ell}(\{f_n\}) = \emptyset$$ and then partition the LHS of that equation to obtain $$\big(\text{START}\triangle\text{Obs}_{\ell}(\{f_1\})\triangle\text{Obs}_{\ell}(\{f_2\})\triangle\cdots\triangle\text{Obs}_{\ell}(\{f_m\})\big)\triangle\big(\text{Obs}_{\ell}(\{f_{m+1}\})\triangle\cdots\triangle\text{Obs}_{\ell}(\{f_n\})\big) = \emptyset$$ where $m \le n$.<br/><br/>
8. Applying a fundamental theorem of symmetric differences that $A \triangle B = \emptyset \Leftrightarrow A = B$, we then conclude $$\big(\text{START}\triangle\text{Obs}_{\ell}(\{f_1\})\triangle\text{Obs}_{\ell}(\{f_2\})\triangle\cdots\triangle\text{Obs}_{\ell}(\{f_m\})\big) = \big(\text{Obs}_{\ell}(\{f_{m+1}\})\triangle\cdots\triangle\text{Obs}_{\ell}(\{f_n\})\big)$$ where we recognize that the LHS (and thus also the RHS by transitivity) is actually $s_k$.<br/><br/>
9. Thus whenever $x \in s_k$, we also have $x \in \big(\text{Obs}_{\ell}(\{f_{m+1}\})\triangle\cdots\triangle\text{Obs}_{\ell}(\{f_n\})\big)$ and thus there must exist some $j \in \{m+1, m+2, \ldots, n\}$ such that $x \in \text{Obs}_{\ell}(\{f_j\})$. That is, there exists a fault $f_j$ among the “remaining” or “unused” faults making up BUFILO $b$ that realizes an outgoing edge from $s_k$ (that is, the observables associated with this fault $f_j$ will cancel the arbitrary observable $x \in s_k$ chosen to be eliminated). At this point, this conclusion means that we can take $s_{k+1} = s_{k}\triangle\text{Obs}_{\ell}(\{f_{j}\})$ and use $(s_0, s_1, \ldots, s_k, s_{k+1})$ as the desired $(k+1)$-length path $p_{k+1}$ whose existence we are trying to establish.

</div>


Recall that we have assumed $P(k)$ is True:

In [ ]:
Pk

And we want to prove $P(k+1)$ is True:

In [ ]:
k_plus_1 = Add(k, one)
Pkplus1 = P.apply(k_plus_1)

We propose the $(k+1)$-length path $p_{k+1} = (s_0, s_1, \ldots, s_k, s_{k+1})$ as the desired path $p$, with $s_{k+1} = s_{k} \triangle \text{Obs}_{\ell}(\{f_j\})$.

We need to show then that $p_{k+1} = (s_0, s_1, \ldots, s_k, s_{k+1})$ is in fact an element of $\text{PathsOf}(G, \text{START}\to)$. To do so, we will eventually invoke and instantiate the `path_with_begin_extension` theorem from the graphs/walks package:

In [ ]:
from proveit.logic.sets import SymmetricDifference
s_k_plus_1 = SymmetricDifference(IndexedVar(s, k), ObservableSet(Set(IndexedVar(f, j)), _ell))

In [ ]:
from proveit.graphs.walks import path_with_begin_extension
path_with_begin_extension

To instantiate that `path_with_begin_extension` theorem, we will use:
1. our BUFILO-generating graph $G_{\ell,, \nu}^{\text{BUF}}$ for graph $G$
2. $k$ for $n$
3. $s_0,s_1,\ldots,s_k$ for $v_0,v_1,...,v_k$ (after the $n:k$ replacement)
4. and $s_{k+1}$ for $w$

In [ ]:
# for item in defaults.assumptions:
#     display(item)

We already have established that $G_{\ell,\nu}^{\text{BUF}}$ is a graph, and by assumption we already have that $k \in \mathbb{N}$:

In [ ]:
buf_gen_graph_is_graph

In [ ]:
from proveit.numbers import Natural
InSet(k, Natural).prove()

We also already have from an earlier Skolemization step that $s_0, s_1, \ldots, s_k \in \text{Vertices}(G_{\ell, \nu}^{\text{BUF}})$, which we can summarize as follows:

In [ ]:
# This is just for display/confirmation and isn't literally needed for the proof process
Forall(i, InSet(IndexedVar(s, i), Vertices(BufiloGeneratingGraph)), domain=Interval(zero, k)).prove()

We still need to establish that our proposed “extension node” $s_{k+1} = s_k \triangle\text{Obs}_{\ell}(\{f_j\}) \in \text{Vertices}(G_{\ell, \nu}^{\text{BUF}})$, which is somewhat round-about over the next few cells:

In [ ]:
# need this to then allow us to show {f_j} in Errors
Forall(i, InSet(IndexedVar(f, i), Faults), domain=Interval(one, n)).prove().instantiate({i:j})

In [ ]:
# need this to then show that Obs({f_j}) in ObsSets
InSet(Set(IndexedVar(f, j)), Errors).prove()

In [ ]:
# Need this to show that Obs({f_j}) in Vertices(G)
obs_set_f_j_in_obs_sets = InSet(ObservableSet(Set(IndexedVar(f, j)), _ell), ObservableSets).prove()

In [ ]:
# Now we can conclude that Obs({f_j}) in Vertices(G). Yay!
# We don't literally need this step for the proof process.
InSet(ObservableSet(Set(IndexedVar(f, j)), _ell), Vertices(BufiloGeneratingGraph)).prove()

In [ ]:
s_k_in_obs_sets = InSet(IndexedVar(s, k), ObservableSets).prove()

In [ ]:
obs_set_f_j_in_obs_sets.unfold()

In [ ]:
s_k_in_obs_sets.unfold()

In [ ]:
from proveit.logic.sets import Union
from proveit.physics.quantum.QEC2 import Detectors
s_k_plus_1.deduce_superset_eq_relation(Union(Detectors, Set(_ell)))

In [ ]:
InSet(s_k_plus_1, ObservableSets).prove()

In [ ]:
# This should now also work but isn't technically required
InSet(s_k_plus_1, Vertices(BufiloGeneratingGraph)).prove()

At last we are ready to instantiate our path-extension theorem:

In [ ]:
path_with_begin_extension

In [ ]:
from proveit import w
path_with_begin_extension.instantiate(
        {G:BufiloGeneratingGraph, n:k, v:s, w:s_k_plus_1},
        assumptions = defaults.assumptions + [LessEq(m, n), InSet(x, s_k)] )

Next, we establish that $\text{Adjacent}(s_k, s_k \triangle \text{OBS}_{\ell}(\{f_{j}\}), G_{\ell, \nu}^{\text{BUF}})$. One way to do this is to establish that $(s_k, s_{k+1}) \in \text{Edges}(G_{\ell, \nu}^{\text{BUF}})$.

In [ ]:
from proveit.graphs import Edges
Edges(BufiloGeneratingGraph)

In [ ]:
from proveit.graphs import edges_def
edges_def

In [ ]:
from proveit import E, V
from proveit.physics.quantum.QEC2 import ObservableSets
edges_of_bug_gen_graph_def = edges_def.instantiate({V:ObservableSets, E:buf_gen_graph_def.rhs.operands[1]})

In [ ]:
edges_of_buf_gen_graph = (
    edges_of_bug_gen_graph_def.inner_expr().lhs.operand.substitute(buf_gen_graph_def.derive_reversed())
)

To establish that $(s_k, s_{k+1}) \in \text{Edges}(G_{\ell, \nu}^{\text{BUF}})$, we need to establish that $\nu(s_k) \in s_{k} - s_{k+1}$. To show _that_, we need to show that $\nu(s_{k}) \in s_{k}$ while $\nu(s_{k}) \notin s_{k+1}$. We already know that $\nu(s_{k}) \in s_{k}$, and that $\nu(s_{k}) \in \text{Obs}_{\ell}(\{f_{j}\})$:

In [ ]:
nu_s_k_in_s_k

In [ ]:
nu_s_k_in_obs_f_j

In [ ]:
nu_s_k_in_obs_f_j.prove()

In [ ]:
from proveit.logic.sets.symmetric_difference import in_both_impl_not_in_sym_diff
in_both_impl_not_in_sym_diff

In [ ]:
s_k_plus_1

In [ ]:
s_k_plus_1.operands[1]

In [ ]:
in_both_impl_not_in_sym_diff.instantiate({A:s_k, B:s_k_plus_1.operands[1], x:Function(_nu, s_k)})

In [ ]:
from proveit.logic.sets import Difference
InSet(Function(_nu, s_k), Difference(s_k, s_k_plus_1)).prove()

In [ ]:
# Concluding membership in a set comprehension continues to be problematic
# InSet(ExprTuple(s_k, s_k_plus_1), edges_of_buf_gen_graph.rhs).prove(auto_simplify=False)

In [ ]:
membership_claim = InSet(ExprTuple(s_k, s_k_plus_1), edges_of_buf_gen_graph.rhs)

In [ ]:
# the simpler 'fold_basic_comprehension' does not apply
from proveit.logic.sets.comprehension import fold
fold

In [ ]:
membership_claim

In [ ]:
from proveit import y, Q
explicit_conditions = membership_claim.domain.explicit_conditions()[0]
_Q_op, _Q_op_sub = (Function(Q, membership_claim.domain.all_instance_vars()),
                    explicit_conditions)
_n_sub = num(2)
_f_op, _f_sub = (Function(f, membership_claim.domain.all_instance_vars()),
                 membership_claim.domain.instance_element)
_S_sub = membership_claim.domain.all_domains()
_x_sub = membership_claim.element
_y_sub = membership_claim.domain.all_instance_vars()
temp_inst = fold.instantiate(
                    {n:_n_sub, S:_S_sub, 
                    _Q_op:_Q_op_sub, _f_op:_f_sub,
                    x:_x_sub, y:_y_sub},
                    # preserved_exprs = {self.domain.condition},
                    auto_simplify = False)

In [ ]:
temp_inst.antecedent.conclude_via_example((s_k, s_k_plus_1))

In [ ]:
s_k_s_k_plus_1_tuple_in_edge_set = temp_inst.derive_consequent()

In [ ]:
InSet(s_k_s_k_plus_1_tuple_in_edge_set.element, edges_of_buf_gen_graph.lhs).prove()

In [ ]:
from proveit.graphs import AdjacentVertices
AdjacentVertices(s_k, s_k_plus_1, BufiloGeneratingGraph).definition()

In [ ]:
# But this won't work because we're dealing with a digraph (with ordered tuples as edges)
# instead of an undirected graph (with 2-elem sets as edges)
from proveit.graphs import AdjacentVertices
try:
    AdjacentVertices(s_k, s_k_plus_1, BufiloGeneratingGraph).definition().derive_left_via_equality()
except Exception as the_exception:
    print(f"EXCEPTION: {the_exception}")

We want to show there exists a path of length $k+1$ that then also has the desired properties.

The path we propose is the path $p' = (s_1, s_2, \ldots, s_k, s_{k+1})$, where:

- $s_1, s_2,\ldots, s_k$ are the nodes from path $p$
- $\text{Adjacent}(s_k, s_{k+1})$ in $G_{\ell, \nu}^{\text{BUF}}$
- $\forall_{i \in \{1..k\}}\big(s_{k+1} \ne s_{i}\big)$

That is, we build path $p'$ by an appropriate extension of the existing path $p$.

In more detail, we have the following initial steps:

1. Show that for any $x \in s_k$ (_i.e._, for any observable in the last node in the "current" assumed path $p$), there exists a fault $f \in b - \{f_1, f_2, \ldots, f_{k-1}\}$ such that $x \in \text{Obs}_{\ell}(\{f\})$.
2. Thus whatever $\nu(s_{k})$ is, there exists a corresponding fault $f$ such that $\nu(s_k) \in \text{Obs}(\{f\})$.
3. Skolemize, letting the existent fault be called $f^{*}$, and let/note $s_{k+1} = s_{k} \triangle \text{Obs}_{\ell}(\{f^{*}\})$.
4. There exists such $s_{k+1} \in \text{Vertices}(G)$ that is adjacent to $s_{k}$.
 - $\text{Adjacent}(s_k, s_{k+1})$ in $G_{\ell, \nu}^{\text{BUF}}$ means $(s_k, s_{k+1}) \in \text{Edges}(G)$
 - $(s_k, s_{k+1}) \in \text{Edges}(G)$ means $(s_k, s_{k+1}) \in \{(s, s')\,|\, s, s' \in G, \nu(s) \in s - s'\}$
 - Show that $\nu(s_{k+1}) \in s_{k} - s_{k+1}$
3. $\forall_{i \in \{1..k\}}\left(s \ne s_{i}\right)$
4. The new sequence $(s_1, s_2, \ldots, s_k, s_{k+1})$ is a path in $G_{\ell, \nu}^{\text{BUF}}$

For Step (3), it would be nice to have a related theorem in the `graphs` package to conclude (3) from (1) and (2) — some sort of “path extension” theorem. 